# 21 · A frozen ImageNet ResNet-18 as the familiarity check (Stage R10)

**Question.** Does a **frozen** ImageNet ResNet-18 embedding detect unfamiliar faults better than the CNN-6 embedding, when
it replaces the shared-covariance Mahalanobis familiarity score and everything else stays the same?

Exploratory, development data only, **no training of any kind**: the 81 saved leave-one-class-out (LOCO) CNN-6 runs (9
withheld classes × 3 outer folds × seeds 42, 43, 44), their saved scores, and frozen ResNet-18 embeddings of the same
windows. Reserved tests stay sealed.

## Why try this

The familiarity check asks: *does this recording look like anything the model learned?* Today it measures that in the
CNN-6 embedding: the distance to the nearest known class. Stage R8 found the weak spot: a withheld **vertical
misalignment** recording sits inside the known clusters (AUROC below 0.5, worse than a coin toss), and so does withheld
**imbalance** (about 0.59). Every distance tried on that embedding failed the same way.

A likely reason: CNN-6 was trained only to tell the *known* classes apart. Whatever makes an unseen fault different was
never useful during training, so the network had no reason to keep it. An encoder trained on something else entirely —
ImageNet photographs — and never shown any of these recordings cannot have learned to throw that information away. If
the difference is still in the signal, a generic encoder might keep it.

**The known risk, stated before any result.** In Stage ML-5 a classifier trained on top of these frozen features reached
only 0.855 macro-F1 on the known classes, and recognised normal recordings badly (F1 0.54). Generic image features may not
separate machine conditions well, and may put an unseen fault next to *normal*, which would make the system call it
healthy. Section "The known risk, measured" checks exactly that.

## The declaration, fixed before any code
Quoted from CLAUDE.md §7 (Stage R10, committed before any code); the cell prints it so the method, arms, metrics, bar and
stop rule are visibly the ones fixed first.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads CLAUDE.md and prints the Stage R10 protocol exactly as committed: encoder, packing, score, data, arms, evaluation, acceptance bar, compute guard, stop rule, and the addendum that settles 27 versus 81 forward passes.</div>

In [ ]:
# Quote the committed Stage R10 declaration from CLAUDE.md.
from ml.paths import SYSTEM_CODE_DIR
claude = (SYSTEM_CODE_DIR.parent / "CLAUDE.md").read_text()
print(claude[claude.index("**Stage R10 protocol"):claude.index("## 7-prev14.")].strip())

## How a window becomes a frozen embedding
No learned adapter: each window's six vibration channels (already normalised as its CNN-6 run normalised them) are packed
into **two colour images**, one per bearing. The inner (underhang) accelerometer's axial, radial and tangential channels
become red, green and blue of image U; the outer (overhang) accelerometer's become image O. Keeping the three directions
of one sensor together preserves the axial / radial / tangential split that carries the misalignment physics; averaging
all six into one grey image would lose it. Each image is stretched to 224 × 224, passed through the frozen network, and
the two 512-number outputs are joined into one 1,024-number embedding.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> draws the packing as a small diagram from one real window (seed-independent run imbalance / fold 0, its first window): the six normalised channels, then the two RGB images U and O that the frozen network sees. It saves the figure to <code>results/figures/r10_packing.png</code>.</div>

In [ ]:
# The packing on one real window: six normalised channels → two RGB images (underhang U, overhang O).
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from ml.config import load_config
from ml.datasets import dev_window_rows, load_representation
from ml.frozen_embed import pack_bearings
from ml.paths import DEFAULT_CONFIG, data_dir, results_dir
from ml.splits import development_pool

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
dc = cfg.development
pool = development_pool(meta, dc.allowed_closed, dc.reserved_columns).merge(
    pd.read_csv(results_dir() / dc.folds_file)[["rec_id", "dev_fold"]], on="rec_id")
arr, index = load_representation(data_dir() / "representations", cfg, meta)
rows = dev_window_rows(index, pool)
z0 = np.load(results_dir() / "scores" / "r18frozen_w-imbalance_loco_fold0_embeddings.npz")
r0, w0 = str(z0["rec_id"][0]), int(z0["window_idx"][0])
norm = json.loads((results_dir() / "loco" / "cnn6_w-imbalance_s42_loco" / "fold0_norm.json").read_text())
x = np.asarray(arr[int(rows[(rows.rec_id == r0) & (rows.window_idx == w0)].row.iloc[0])], dtype=np.float32)[:6]
x = (x - np.asarray(norm["mean"])[:, None, None]) / np.asarray(norm["std"])[:, None, None]
u, o = pack_bearings(torch.from_numpy(x[None]))
fig, ax = plt.subplots(1, 8, figsize=(13, 2.6), gridspec_kw={"width_ratios": [1] * 6 + [1.6, 1.6]})
for i, name in enumerate(norm["channels"]):
    ax[i].imshow(x[i], aspect="auto", origin="lower", cmap="gray"); ax[i].set_title(name.replace("_", "\n"), fontsize=8)
rgb = lambda t: ((t[0].permute(1, 2, 0).numpy() - t.min().item()) / (t.max() - t.min()).item())
ax[6].imshow(rgb(u), aspect="auto", origin="lower"); ax[6].set_title("image U (R,G,B =\nunder ax, rad, tan)", fontsize=8)
ax[7].imshow(rgb(o), aspect="auto", origin="lower"); ax[7].set_title("image O (R,G,B =\nover ax, rad, tan)", fontsize=8)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
ax[0].set_ylabel("order bin (0.5–150 orders, log)\n× frame (time)", fontsize=8)
fig.suptitle(f"One window ({r0}, window {w0}): six channels → two RGB images → frozen ResNet-18 → 2 × 512 numbers", fontsize=9)
fig.tight_layout(); fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r10_packing.png", dpi=150); plt.show()

## Load the runs and the forward-pass record
The embeddings were computed by `scripts/38_frozen_embeddings.py` (forward passes only). The 3C normalisation and the
window tables are identical across the three seeds for every (class, fold), so each (class, fold) was embedded once — 27
passes — and reused for all seeds; the script re-checked that before each pass.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints provenance (git sha, split hash, config sha, the ImageNet weights sha256) and the forward-pass record written by the script: device, total time, the projection after the first pass, and the per-(class, fold) check that the normalisation and window table are the same for all three seeds.</div>

In [ ]:
# Provenance and the forward-pass record (27 passes, seed independence re-checked per pass).
from ml.config import config_sha256
from ml.frozen_embed import embedding_path, nearest_normal_share, run_scores_f
from ml.loco_eval import decide, macro, recordings
from ml.models.resnet import load_resnet_config, verify_weights
from ml.rmd_eval import evaluate_run, gate_ok, pooled, reading, reproduction, rule_b, runs, thresholds
from ml.splits import FAULT_CLASSES, split_hash
from ml.utils import git_sha

prov = {"git_sha": git_sha(), "split_hash": split_hash(meta), "config_sha256": config_sha256(cfg),
        "weights_sha256": verify_weights(load_resnet_config())}
print(prov)
fwd = json.loads((results_dir() / "research" / "r10_embedding_run.json").read_text())
si = pd.DataFrame(fwd["seed_independence"])
print(f"device {fwd['device']}; {len(fwd['passes'])} passes, {fwd['total_seconds']} s in all "
      f"(first pass {fwd['first_pass_seconds']} s, projected {fwd['projected_total_minutes']} min); "
      f"normalisation identical across seeds in {si.norm_identical.sum()}/27, window table in {si.table_identical.sum()}/27")
assert len(fwd["passes"]) == 27 and si.norm_identical.all() and si.table_identical.all()
ARMS2 = ("M", "F")
ARM_NAMES = {"M": "CNN-6 embedding, shared Mahalanobis (current)", "F": "frozen ResNet-18 embedding, shared Mahalanobis (new)"}
plan = runs()
print(len(plan), "runs")

## Reproduction and alignment check (before anything else)
Three checks, each of which stops the notebook on failure: (1) the frozen embeddings line up row for row with the saved
CNN-6 window table (inside `run_scores_f`); (2) arm M recomputed from the saved CNN-6 embeddings equals the saved Stage 3C
Mahalanobis (|Δ maha| ≤ 1e-4 × maha), and its per-run results equal R8's arm M in `r8_per_run.csv`; (3) arm M's decisions
equal `ml.loco_eval.decide`'s rule B on every recording.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> scores every one of the 81 runs (arm M from the saved CNN-6 embeddings, arm F from the frozen embeddings, plus each recording's nearest class centroid under both), then runs the three checks. The asserts stop the notebook on any mismatch; the printed numbers are the largest differences found.</div>

In [ ]:
# Score the 81 runs; alignment (inside run_scores_f), M vs saved 3C maha, M vs R8's per-run results, M vs decide's rule B.
import time
t0 = time.perf_counter()
tables, fits, repro = {}, {}, []
for s, c, k in plan:
    rec, info = run_scores_f(s, c, k)
    tables[(s, c, k)], fits[(s, c, k)] = rec, info
    repro.append({"seed": s, "withheld": c, "fold": k, "max_rel_diff_maha": reproduction(rec)["max_rel_diff_maha"]})
repro = pd.DataFrame(repro)
score_seconds = time.perf_counter() - t0
rows_ = []
for (s, c, k), rec in tables.items():
    for r in evaluate_run(rec, ARMS2):
        rows_.append({"seed": s, "withheld": c, "outer_fold": k, **r})
per_run = pd.DataFrame(rows_)
r8 = pd.read_csv(results_dir() / "research" / "r8_per_run.csv").query("arm == 'M'").set_index(["seed", "withheld", "outer_fold"])
mine = per_run.query("arm == 'M'").set_index(["seed", "withheld", "outer_fold"]).loc[r8.index]
cols = [c for c in r8.columns if c != "arm"]
r8_diff = float(np.nanmax(np.abs(mine[cols].to_numpy(float) - r8[cols].to_numpy(float))))
dec_mismatch = 0
for s in (42, 43, 44):
    d = decide(recordings(s)).set_index(["withheld", "outer_fold", "rec_id"])
    for c in FAULT_CLASSES:
        for k in (0, 1, 2):
            rec = tables[(s, c, k)]
            t = thresholds(rec, ARMS2)
            mine_d = rule_b(rec.pred, gate_ok(rec.msp, rec.M, t["tau_p"], t["tau_M"]))
            dec_mismatch += int((d.loc[(c, k)].loc[rec.rec_id].decision_B.to_numpy() != mine_d).sum())
print(f"scored 81 runs in {score_seconds:.0f} s")
print(f"max relative |Δ maha| (M vs saved 3C) = {repro.max_rel_diff_maha.max():.2e}")
print(f"max |Δ| of arm M's per-run results vs R8's r8_per_run.csv = {r8_diff:.2e}")
print(f"recordings where arm M's decision differs from decide's rule B: {dec_mismatch}")
assert (repro.max_rel_diff_maha <= 1e-4).all() and r8_diff <= 1e-9 and dec_mismatch == 0

## The two scores on one run (sanity table)
One run (seed 42, withheld imbalance, fold 0): a few recordings of each role with M (the CNN-6 familiarity distance) and F
(the frozen-embedding distance). Both are distances to the nearest known class: higher = more unfamiliar. Their scales
differ (512-d against 1,024-d), which does not matter because every threshold is set per arm.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows, for one run, three recordings of each evaluated role with the saved Mahalanobis, arm M, arm F and the nearest class under each, and the Ledoit–Wolf shrinkage of both fits.</div>

In [ ]:
# Sanity view of one run.
one = tables[(42, "imbalance", 0)]
print({k: v for k, v in fits[(42, "imbalance", 0)].items() if k in ("shrinkage_m", "shrinkage_f", "n_fit_windows")})
one.groupby("role").head(3)[["rec_id", "role", "class10", "pred", "msp", "maha", "M", "F", "nearest_M", "nearest_F"]].round(4)

## Threshold-free results
Per run, AUROC and OSCR with the withheld-class recordings of fold k as positives and fold k's known recordings as
negatives (weight 1, OSCR correctness = CNN-6 top class is the true class). The macro is the mean over folds within a
withheld class, then over the 9 classes, per seed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> tabulates the macro AUROC and OSCR per seed for M and F, with the mean over the three seeds (the per-run metrics were computed above by <code>ml.rmd_eval.evaluate_run</code>).</div>

In [ ]:
# Macro AUROC / OSCR per seed and arm, and the seed mean.
tf = pd.DataFrame([{"seed": s, "arm": a, "macro_auroc": macro(g, "auroc"), "macro_oscr": macro(g, "oscr")}
                   for (s, a), g in per_run.groupby(["seed", "arm"])])
tf_table = tf.pivot(index="seed", columns="arm", values=["macro_auroc", "macro_oscr"])
tf_table.loc["mean"] = tf_table.mean()
tf_table.round(4)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> draws the per-class AUROC (mean over folds and seeds) for M and F as grouped bars and saves it to <code>results/figures/r10_auroc_per_class.png</code>. A bar below the dashed 0.5 line means the score ranks that withheld class as more familiar than the known recordings.</div>

In [ ]:
# Per-class AUROC, mean over folds and seeds.
pc = per_run.groupby(["withheld", "arm"]).auroc.mean().unstack()[list(ARMS2)]
fig, ax = plt.subplots(figsize=(10, 3.8))
pc.plot.bar(ax=ax, width=.75, color=["C0", "C3"])
ax.axhline(0.5, color="k", lw=.8, ls="--")
ax.set_ylabel("AUROC (withheld vs fold-k known)"); ax.set_xlabel("withheld class"); ax.set_ylim(0.3, 1.0)
ax.set_title("Per-class AUROC, mean over folds and seeds"); ax.legend([ARM_NAMES[a] for a in ARMS2], fontsize=8, loc="lower left")
plt.xticks(rotation=30, ha="right"); fig.tight_layout()
fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r10_auroc_per_class.png", dpi=150); plt.show()
pc.round(3)

## In the decision rule (the primary result)
F replaces the shared Mahalanobis term in the confidence gate; τ_p, rule B and the predicted class are unchanged.
**Primary:** τ_s at a matched known-side rejection count — F's gate rejects exactly as many fold-k known recordings as M's
gate does at τ_d. Pooled over the 27 runs of each seed: unknown recall, withheld recordings called healthy (must not rise)
and known recordings called the wrong fault.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> pools the matched-threshold outcomes over the 27 runs of each seed, per arm: unknown recall (withheld recordings sent to further inspection), withheld called healthy, known called the wrong fault, and the known-side rejections (equal for both arms by construction, unless a count could not be reached; that number is printed first).</div>

In [ ]:
# Matched known-side rejection count: pooled outcomes per seed and arm.
unreached = per_run[per_run.tau_matched.isna()]
print("runs where the matched count could not be reached:", len(unreached))
matched = pooled(per_run, "matched")
matched.round(4)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows the same outcomes with τ_s set the way τ_d is set today (the 0.975 order statistic of the calibration recordings): what each arm would do in practice, where the known-side rejections are no longer equal.</div>

In [ ]:
# Secondary: τ_s calibrated like τ_d.
calibrated = pooled(per_run, "calibrated")
calibrated.round(4)

## Per-class unknown recall, M against F
Unknown recall at the matched count per withheld class (pooled over the class's 3 folds within a seed, then the mean over
seeds); classes where F is better or worse by 5 points or more are marked.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes unknown recall per withheld class for both arms and every seed at the matched threshold, takes the seed mean, marks ≥ 5-point differences, and draws M and F side by side (<code>results/figures/r10_recall_per_class.png</code>).</div>

In [ ]:
# Per-class unknown recall (matched count): per seed, seed mean, F − M with ≥ 5-point differences marked.
g = per_run.groupby(["seed", "withheld", "arm"])
cls = (g.matched_withheld_unknown.sum() / g.matched_withheld_n.sum()).unstack("arm")
cls_mean = cls.groupby(level="withheld").mean()
cls_mean["F_minus_M"] = cls_mean.F - cls_mean.M
cls_mean["mark"] = np.where(cls_mean.F_minus_M >= 0.05, "F better (≥ 5 pts)", np.where(cls_mean.F_minus_M <= -0.05, "F worse (≥ 5 pts)", ""))
fig, ax = plt.subplots(figsize=(10, 3.6))
cls_mean[["M", "F"]].plot.bar(ax=ax, width=.75, color=["C0", "C3"])
ax.set_ylabel("unknown recall (matched count)"); ax.set_xlabel("withheld class"); ax.set_ylim(0, 1)
ax.set_title("Unknown recall per withheld class, mean over seeds"); ax.legend([ARM_NAMES["M"], ARM_NAMES["F"]], fontsize=8)
plt.xticks(rotation=30, ha="right"); fig.tight_layout()
fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r10_recall_per_class.png", dpi=150); plt.show()
display(cls.unstack("seed")[["M", "F"]].round(3))
cls_mean.round(3)

## The known risk, measured
Descriptive only; it does not enter the reading. For each withheld class: the share of its withheld recordings whose
nearest known-class centroid is **normal** (by the mean over the recording's windows of the distance to each centroid),
under M and under F. A higher share under F would mean the frozen features place the unseen fault next to healthy.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes, per seed and withheld class, the share of withheld recordings whose nearest class centroid is normal under M and under F, and shows the seed mean per class.</div>

In [ ]:
# Nearest-centroid-is-normal shares per withheld class (M vs F), and the seed mean.
per_rec = pd.concat([t.assign(seed=s, withheld=c, outer_fold=k) for (s, c, k), t in tables.items()], ignore_index=True)
nn_share = nearest_normal_share(per_rec)
nn_mean = nn_share.groupby("withheld")[["share_normal_M", "share_normal_F"]].mean()
nn_mean.loc["all classes (mean)"] = nn_mean.mean()
nn_mean.round(3)

## Reading against the declared bar
F **helps, confirmed** only if, against M at the matched count, unknown recall is at least 5 points higher AND no more
withheld recordings are called healthy, on all three seeds; otherwise **worse** if the mean change is negative; otherwise
**does not help**. Applied mechanically:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the declared bar to the matched-count table (<code>ml.rmd_eval.reading</code> with arm F) and prints, per seed, the recall gain of F over M, the change in withheld recordings called healthy, whether the bar is met, and the reading.</div>

In [ ]:
# The declared reading, applied mechanically.
verdict = reading(matched, "F")
for s, v in verdict["per_seed"].items():
    print(f"seed {s}: unknown recall F − M = {v['recall_gain'] * 100:+.1f} points; withheld called healthy F − M = "
          f"{v['healthy_change']:+d}; bar met: {v['bar_met']}")
print(f"mean recall change {verdict['mean_recall_gain'] * 100:+.1f} points  →  READING: {verdict['reading'].upper()}")

**In plain words.** _(written after the run, from the numbers above)_

## Limitations
- Development data only; nothing here is an independent claim (reserved tests sealed; the final evaluation already ran).
- The thresholds come from in-sample calibration recordings of each run; the matched comparison removes threshold choice
  from the primary result but uses the same known recordings that are counted.
- The same LOCO partition, folds and saved CNN-6 runs as Stages 3C and R4–R9; every withheld class was seen during earlier
  development of the system.
- One frozen encoder, one fixed packing and one estimator, with nothing tuned; a negative result says this generic
  embedding does not help here, not that no unsupervised or self-supervised representation could.
- Each (class, fold) was embedded once and reused for all seeds, so arm F varies across seeds only through which
  recordings CNN-6 predicts and τ_p (the CNN-6 parts of the gate), not through the embedding.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the per-run table to <code>results/research/r10_per_run.csv</code> and every reported number, with provenance and the reading, to <code>results/research/r10_frozen_resnet.json</code>.</div>

In [ ]:
# Write the outputs: per-run CSV and the JSON summary.
out = results_dir() / "research"
per_run.to_csv(out / "r10_per_run.csv", index=False)
summary = {"stage": "R10", "provenance": prov | {"score_seconds": round(score_seconds, 1)},
           "forward_passes": {k: fwd[k] for k in ("device", "total_seconds", "first_pass_seconds", "projected_total_minutes", "finding")}
                             | {"n_passes": len(fwd["passes"])},
           "arms": ARM_NAMES,
           "reproduction": {"max_rel_diff_maha": float(repro.max_rel_diff_maha.max()), "max_abs_diff_vs_r8_per_run": r8_diff,
                            "decision_mismatches_vs_decide_rule_B": dec_mismatch},
           "threshold_free_macro": tf.to_dict(orient="records"),
           "threshold_free_per_class": per_run.groupby(["seed", "withheld", "arm"])[["auroc", "oscr"]].mean().reset_index().to_dict(orient="records"),
           "matched": matched.to_dict(orient="records"), "calibrated": calibrated.to_dict(orient="records"),
           "matched_unreached_runs": int(len(unreached)),
           "recall_per_class_matched": cls.reset_index().to_dict(orient="records"),
           "recall_per_class_seed_mean": cls_mean.reset_index().to_dict(orient="records"),
           "nearest_is_normal": nn_share.to_dict(orient="records"),
           "fits": {f"{s}/{c}/{k}": v for (s, c, k), v in fits.items()}, "reading": verdict}
(out / "r10_frozen_resnet.json").write_text(json.dumps(summary, indent=1, default=float))
print("written:", (out / "r10_per_run.csv").relative_to(SYSTEM_CODE_DIR), (out / "r10_frozen_resnet.json").relative_to(SYSTEM_CODE_DIR))

## Outputs written
- `results/research/r10_frozen_resnet.json`, `results/research/r10_per_run.csv`
- Figures: `results/figures/r10_packing.png`, `r10_auroc_per_class.png`, `r10_recall_per_class.png`
- Embeddings (gitignored): `results/scores/r18frozen_w-<c>_loco_fold<k>_embeddings.npz`, written by
  `scripts/38_frozen_embeddings.py`, with its record `results/research/r10_embedding_run.json`